# Notebook 03 — Transformación según reglas de negocio

Tercer sub-bloque del Tema 04. Con los DataFrames ya **limpios y tipados** del Notebook 02, ahora los **reorganizas** hacia el modelo dimensional: dimensiones desnormalizadas, `dim_date` generada en pandas, tabla de hechos con surrogate keys resueltas vía merge sucesivo.

Este notebook es el equivalente programático de los `INSERT…SELECT…JOIN` que pueblan `northwind_dwh` en los scripts SQL del repo. Al terminar deberías tener en memoria seis DataFrames listos para cargar al DWH (las 5 dims + la fact).

## Setup — recuperar DataFrames limpios del Notebook 02

Re-creación del engine, extracción de las **8 tablas** del OLTP que necesitamos para el modelado dimensional (las 5 que ya conocías + `categories`, `suppliers` y `shippers` que entran en los joins de las dimensiones), y las correcciones de tipo que dejaste resueltas en el Notebook 02 (fechas a `datetime`, dinero redondeado a 2 decimales).

Notebook auto-contenido: este bloque te deja en el mismo estado en el que terminaste el 02.

In [ ]:
import pandas as pd
from sqlalchemy import create_engine

# Reemplaza con tus valores del Tema 01
AURORA_HOST     = "aurora-mod4.cluster-xxxxx.us-east-1.rds.amazonaws.com"
AURORA_PASSWORD = "TU_PASSWORD_AQUI"
AURORA_DATABASE = "northwind"

engine = create_engine(
    f"postgresql+psycopg2://postgres:{AURORA_PASSWORD}@{AURORA_HOST}:5432/{AURORA_DATABASE}"
)

# Extracción de las 8 tablas OLTP necesarias
df_customers     = pd.read_sql("SELECT * FROM northwind_oltp.customers",     engine)
df_orders        = pd.read_sql("SELECT * FROM northwind_oltp.orders",        engine)
df_order_details = pd.read_sql("SELECT * FROM northwind_oltp.order_details", engine)
df_products      = pd.read_sql("SELECT * FROM northwind_oltp.products",      engine)
df_categories    = pd.read_sql("SELECT * FROM northwind_oltp.categories",    engine)
df_suppliers     = pd.read_sql("SELECT * FROM northwind_oltp.suppliers",     engine)
df_employees     = pd.read_sql("SELECT * FROM northwind_oltp.employees",     engine)
df_shippers      = pd.read_sql("SELECT * FROM northwind_oltp.shippers",      engine)

# Correcciones de tipo (resultado del Notebook 02)
df_orders["order_date"]    = pd.to_datetime(df_orders["order_date"],    errors="coerce")
df_orders["required_date"] = pd.to_datetime(df_orders["required_date"], errors="coerce")
df_orders["shipped_date"]  = pd.to_datetime(df_orders["shipped_date"],  errors="coerce")

df_order_details["unit_price"] = df_order_details["unit_price"].round(2)
df_order_details["discount"]   = df_order_details["discount"].round(2)

print(f"customers       {df_customers.shape}")
print(f"orders          {df_orders.shape}")
print(f"order_details   {df_order_details.shape}")
print(f"products        {df_products.shape}")
print(f"categories      {df_categories.shape}")
print(f"suppliers       {df_suppliers.shape}")
print(f"employees       {df_employees.shape}")
print(f"shippers        {df_shippers.shape}")

## Cálculo de valores derivados

En el Tema 02 viste que `fact_sales` declara `extended_price` y `line_total` como **`GENERATED ALWAYS AS … STORED`** — el motor las mantiene automáticamente, no las insertas tú. En Python no existe esa cláusula declarativa, así que las calculas explícitamente en pandas.

La clave es que pandas **vectoriza** las operaciones: cuando escribes `df['a'] * df['b']`, NumPy aplica la multiplicación a todo el array de una sola pasada, sin loops Python. Es entre 50 y 100× más rápido que iterar fila por fila, y la sintaxis es declarativa — describes *qué* quieres, no *cómo* iterarlo.

In [ ]:
# Equivalente a las generated columns del DDL de fact_sales:
#   extended_price = quantity * unit_price
#   line_total     = quantity * unit_price * (1 - discount)

df_order_details["extended_price"] = (
    df_order_details["quantity"] * df_order_details["unit_price"]
).round(2)

df_order_details["line_total"] = (
    df_order_details["quantity"]
    * df_order_details["unit_price"]
    * (1 - df_order_details["discount"])
).round(2)

df_order_details[[
    "order_id", "product_id", "quantity", "unit_price", "discount",
    "extended_price", "line_total"
]].head()

Como contraste, evita el loop fila-por-fila. Funciona, pero es órdenes de magnitud más lento y oculta la intención:

```python
# ❌ Anti-patrón: itera 2 155 veces, lento y verbose
for i in range(len(df_order_details)):
    df_order_details.loc[i, 'line_total'] = (
        df_order_details.loc[i, 'quantity']
        * df_order_details.loc[i, 'unit_price']
        * (1 - df_order_details.loc[i, 'discount'])
    )
```

**Regla:** si te ves escribiendo un `for` sobre filas de un DataFrame, casi siempre hay una versión vectorizada (operación columna a columna, `.apply`, `np.where`, `pd.merge`, …).

In [ ]:
# Spot check rápido — total de ventas netas
total = df_order_details["line_total"].sum()
print(f"SUM(line_total) en pandas: {total:,.2f}")

## Joins entre DataFrames con `merge`

`pd.merge` es la versión pandas de `JOIN`. Mismas semánticas (`inner`, `left`, `right`, `outer`), distinta sintaxis. Piensa el merge como un `JOIN ... ON` con un lado izquierdo y un lado derecho, y mantén el control de qué columna empareja cuál.

**Sintaxis básica:**

```python
pd.merge(izq, der, on='col', how='inner')
# o como método:
izq.merge(der, on='col', how='inner')
```

**Cuando los nombres de columna difieren** entre los dos lados:

```python
izq.merge(der, left_on='customer_id', right_on='id_cliente', how='left')
```

**Cuidados clave:**

- Si la key del lado derecho **no es única**, multiplicas filas del izquierdo — mismo fenómeno que el *fan trap* en SQL. pandas trae el parámetro `validate=` para detectarlo antes de que pase.
- **`how='inner'`** descarta las filas que no matchean en ninguno de los dos lados. **`how='left'`** preserva todas las filas del lado izquierdo aunque no haya match (los campos del derecho quedan en `NaN`). **`how='outer'`** conserva las filas de ambos lados — las del izquierdo que no matchean traen `NaN` en las columnas del derecho, y viceversa. Es la opción para comparar dos fuentes que *deberían* coincidir y detectar ausencias en cualquiera de las dos. (`how='right'` existe pero se usa rara vez; invertir los lados del merge suele leer mejor.)
- Si una columna existe en ambos lados pero **no es la key**, pandas la sufija con `_x` / `_y`. Renombrar o seleccionar antes del merge evita confusión.

In [ ]:
# Equivalente a:
#   SELECT od.*, p.product_name, c.category_name
#   FROM order_details od
#   JOIN products   p ON od.product_id  = p.product_id
#   JOIN categories c ON p.category_id  = c.category_id

df_od_enriched = (
    df_order_details
    .merge(
        df_products[["product_id", "product_name", "category_id"]],
        on="product_id", how="inner", validate="many_to_one",
    )
    .merge(
        df_categories[["category_id", "category_name"]],
        on="category_id", how="inner", validate="many_to_one",
    )
)

df_od_enriched[[
    "order_id", "product_name", "category_name", "quantity", "line_total"
]].head()

El parámetro `validate="many_to_one"` te protege: pandas verifica que la key del lado derecho sea única; si no lo es, **falla con `MergeError` en vez de duplicar filas silenciosamente**. Es una de las defensas más útiles cuando armas un ETL — un fan trap puede pasar desapercibido hasta que los totales no cuadran.

Los cuatro valores válidos:

| `validate=` | Significado |
|---|---|
| `"one_to_one"` | Key única en ambos lados |
| `"one_to_many"` | Key única solo en el izquierdo |
| `"many_to_one"` | Key única solo en el derecho (el caso típico al enriquecer con un catálogo) |
| `"many_to_many"` | Sin restricción de unicidad (rara vez es intencional — más bien señal de que la lógica del merge no está clara) |

## Generación de `dim_date` en pandas

Recuerda del Tema 02: la dimensión de fecha **se genera**, no se extrae. Una fila por día en el rango que cubra todas las fechas del OLTP. Northwind va de 1996-07 a 1998-05, así que un rango 1996-01-01 → 1998-12-31 cubre con margen.

Tres pasos:

1. **`pd.date_range`** genera la secuencia de fechas (la equivalente pandas de `generate_series` en SQL).
2. Los accesores **`.dt`** exponen año, mes, día de la semana, etc. directamente sobre la serie.
3. La **smart key** (`YYYYMMDD` como entero) se calcula sin merge — solo aritmética.

Patrón Kimball: la `dim_date` se pre-popula **densamente** (incluye sábados, domingos y días sin actividad). Eso permite que las queries con `OUTER JOIN` muestren ceros legítimos en huecos del calendario, en vez de simplemente omitir esos días.

In [ ]:
# 1. Rango denso de fechas (una por día)
fechas = pd.date_range(start="1996-01-01", end="1998-12-31", freq="D")
dim_date = pd.DataFrame({"full_date": fechas})

# 2. Smart key YYYYMMDD como entero (la PK no es surrogate, es derivada)
dim_date["date_key"] = (
    dim_date["full_date"].dt.year * 10000
    + dim_date["full_date"].dt.month * 100
    + dim_date["full_date"].dt.day
)

# 3. Atributos derivados — todos vienen del accesor .dt
dim_date["year"]               = dim_date["full_date"].dt.year
dim_date["quarter"]            = dim_date["full_date"].dt.quarter
dim_date["month_number"]       = dim_date["full_date"].dt.month
dim_date["week_of_year"]       = dim_date["full_date"].dt.isocalendar().week.astype(int)
dim_date["day_of_month"]       = dim_date["full_date"].dt.day
dim_date["day_of_week_number"] = dim_date["full_date"].dt.dayofweek + 1  # ISO: lunes=1
dim_date["is_weekend"]         = dim_date["day_of_week_number"].isin([6, 7])

# Nombres en español sin depender del locale del sistema
meses = ["Enero", "Febrero", "Marzo", "Abril", "Mayo", "Junio",
         "Julio", "Agosto", "Septiembre", "Octubre", "Noviembre", "Diciembre"]
dias  = ["Lunes", "Martes", "Miércoles", "Jueves", "Viernes", "Sábado", "Domingo"]

dim_date["month_name"]       = dim_date["month_number"].map(lambda m: meses[m-1])
dim_date["day_of_week_name"] = dim_date["day_of_week_number"].map(lambda d: dias[d-1])

# Reordenar columnas para que coincidan con el DDL
dim_date = dim_date[[
    "date_key", "full_date", "year", "quarter",
    "month_number", "month_name", "week_of_year",
    "day_of_month", "day_of_week_number", "day_of_week_name",
    "is_weekend"
]]

print(f"dim_date: {dim_date.shape}")
dim_date.head()

Notas de implementación:

- **`dt.dayofweek`** devuelve 0..6 con lunes=0 (convención Python). Sumando 1 obtienes 1..7 con lunes=1 (ISO 8601), que es la convención del DDL.
- **Nombres por `.map` con una lista**, no por `locale` del sistema operativo, para que el código sea **portable**: alguien corriendo Windows en inglés obtiene los mismos nombres en español sin configurar nada.
- `dim_date` se genera **en memoria sin merge** con nadie — es independiente del OLTP. Eso permite tenerla disponible incluso si los datos transaccionales todavía no se han cargado.

## Construcción de las dimensiones desnormalizadas

Cada dimensión necesita tres cosas: (a) las columnas correctas según el DDL, (b) las desnormalizaciones que correspondan (aplanar jerarquías, traer atributos de tablas vecinas), (c) una **surrogate key** secuencial (1, 2, 3, …) que no existe en el OLTP.

**Patrón de surrogate key en pandas:**

```python
dim_xxx = dim_xxx.reset_index(drop=True)
dim_xxx.insert(0, 'xxx_key', dim_xxx.index + 1)
```

`reset_index(drop=True)` reinicia el índice a `0..N-1`; sumando 1 obtienes `1..N` como en el DDL (`GENERATED ALWAYS AS IDENTITY`). `insert(0, ...)` coloca la nueva columna al inicio, no al final.

In [ ]:
# dim_customer — mapeo directo, sin merges
dim_customer = (
    df_customers[[
        "customer_id", "company_name", "contact_name", "contact_title",
        "city", "region", "postal_code", "country"
    ]]
    .reset_index(drop=True)
)
dim_customer.insert(0, "customer_key", dim_customer.index + 1)

print(f"dim_customer: {dim_customer.shape}")
dim_customer.head(3)

**`dim_product`** sí necesita dos merges — aplanar `products + categories + suppliers` en una sola tabla. Es la desnormalización que viste en el Tema 02: el DWH viola 3NF deliberadamente para evitar el JOIN extra en cada query analítica.

In [ ]:
# dim_product — aplana products + categories + suppliers
dim_product = (
    df_products[[
        "product_id", "product_name", "category_id", "supplier_id", "discontinued"
    ]]
    .merge(
        df_categories[["category_id", "category_name", "description"]]
            .rename(columns={"description": "category_desc"}),
        on="category_id", how="inner", validate="many_to_one",
    )
    .merge(
        df_suppliers[["supplier_id", "company_name", "country", "city"]]
            .rename(columns={
                "company_name": "supplier_name",
                "country":      "supplier_country",
                "city":         "supplier_city",
            }),
        on="supplier_id", how="inner", validate="many_to_one",
    )
)

# Cast a BOOLEAN — el OLTP guarda smallint 0/1
dim_product["discontinued"] = dim_product["discontinued"].astype(bool)

# Reordenar columnas según DDL
dim_product = dim_product[[
    "product_id", "product_name",
    "category_id", "category_name", "category_desc",
    "supplier_id", "supplier_name", "supplier_country", "supplier_city",
    "discontinued"
]].reset_index(drop=True)
dim_product.insert(0, "product_key", dim_product.index + 1)

print(f"dim_product: {dim_product.shape}")
dim_product.head(3)

**`dim_employee`** requiere un **self-merge** para resolver la auto-referencia: `reports_to` en el OLTP es un FK a la propia tabla `employees`. Lo aplanamos a `reports_to_name` (string) para que el DWH no tenga la auto-referencia.

Usamos `how='left'`: el CEO de Northwind tiene `reports_to IS NULL` (no le reporta a nadie), y un `inner` lo descartaría — perderíamos la raíz del árbol. Patrón general en jerarquías auto-referenciales: **`left join` al hacer self-merge para preservar la raíz**.

In [ ]:
# dim_employee — full_name concatenado + reports_to_name vía self-merge

# Lado izquierdo: el empleado
empleado = df_employees[[
    "employee_id", "first_name", "last_name", "title", "city",
    "country", "region", "hire_date", "reports_to"
]].copy()
empleado["full_name"] = empleado["first_name"] + " " + empleado["last_name"]

# Lado derecho: el manager (misma tabla, distinto rol)
manager = df_employees[["employee_id", "first_name", "last_name"]].copy()
manager["reports_to_name"] = manager["first_name"] + " " + manager["last_name"]
manager = manager[["employee_id", "reports_to_name"]].rename(
    columns={"employee_id": "manager_id"}
)

# how='left' preserva al CEO (reports_to IS NULL en el origen)
dim_employee = empleado.merge(
    manager,
    left_on="reports_to", right_on="manager_id",
    how="left",
)

# Seleccionar columnas finales según DDL
dim_employee = dim_employee[[
    "employee_id", "full_name", "title", "city", "country", "region",
    "hire_date", "reports_to_name"
]].reset_index(drop=True)
dim_employee.insert(0, "employee_key", dim_employee.index + 1)

print(f"dim_employee: {dim_employee.shape}")
dim_employee

Fíjate en la fila del **CEO** (Andrew Fuller): `reports_to_name` queda como `NaN`. Eso es exactamente lo que queremos — el CEO no le reporta a nadie. Si hubiéramos usado `how='inner'`, esa fila desaparecería de `dim_employee` y la fact perdería todas las ventas atribuidas al CEO.

In [ ]:
# dim_shipper — mapeo directo
dim_shipper = (
    df_shippers[["shipper_id", "company_name", "phone"]]
    .reset_index(drop=True)
)
dim_shipper.insert(0, "shipper_key", dim_shipper.index + 1)

print(f"dim_shipper: {dim_shipper.shape}")
dim_shipper

## Construcción de `fact_sales`

La fact se construye **al final** porque depende de que las cinco dimensiones ya tengan sus surrogate keys asignadas. La estrategia es la misma del script `04_fact_populate.sql`:

1. Partir de `order_details` — es el grano del fact (una fila = una línea de pedido).
2. Joinear con `orders` para traer `customer_id`, `employee_id`, `ship_via`, y las tres fechas.
3. **Merge sucesivo** con cada dimensión para resolver cada natural key (`customer_id`, `product_id`, `employee_id`, `ship_via`) a su surrogate key correspondiente.
4. Las **tres date_keys** se calculan directamente desde las fechas, sin merge contra `dim_date` — la smart key `YYYYMMDD` es derivable.
5. Conservar `order_id` como **degenerate dimension** (vive en la fact, no tiene tabla propia).
6. Asignar `sale_key` secuencial al cierre.

In [ ]:
# 1. Partimos de order_details — el grano del fact
fact = df_order_details[[
    "order_id", "product_id", "quantity", "unit_price", "discount",
    "extended_price", "line_total"
]].copy()

# 2. Join con orders para traer customer_id, employee_id, ship_via, fechas
fact = fact.merge(
    df_orders[[
        "order_id", "customer_id", "employee_id", "ship_via",
        "order_date", "required_date", "shipped_date"
    ]],
    on="order_id", how="inner", validate="many_to_one",
)

# 3. Resolver natural keys → surrogate keys con merges sucesivos
fact = fact.merge(
    dim_customer[["customer_id", "customer_key"]],
    on="customer_id", how="inner", validate="many_to_one",
)
fact = fact.merge(
    dim_product[["product_id", "product_key"]],
    on="product_id", how="inner", validate="many_to_one",
)
fact = fact.merge(
    dim_employee[["employee_id", "employee_key"]],
    on="employee_id", how="inner", validate="many_to_one",
)
# how='left' porque shipper_key puede ser NULL en el patrón general
fact = fact.merge(
    dim_shipper[["shipper_id", "shipper_key"]],
    left_on="ship_via", right_on="shipper_id", how="left",
)

# 4. Tres date_keys calculadas directamente desde las fechas
def to_date_key(series: pd.Series) -> pd.Series:
    """Convierte una serie de fechas a smart key YYYYMMDD (Int64 nullable)."""
    return (series.dt.year * 10000
            + series.dt.month * 100
            + series.dt.day).astype("Int64")

fact["order_date_key"]    = to_date_key(fact["order_date"])
fact["required_date_key"] = to_date_key(fact["required_date"])
fact["shipped_date_key"]  = to_date_key(fact["shipped_date"])   # <NA> donde shipped_date es NaT

# 5. Seleccionar columnas finales según el DDL
fact_sales = fact[[
    "order_id",
    "customer_key", "product_key", "employee_key", "shipper_key",
    "order_date_key", "required_date_key", "shipped_date_key",
    "quantity", "unit_price", "discount",
    "extended_price", "line_total",
]].reset_index(drop=True)

# 6. sale_key secuencial
fact_sales.insert(0, "sale_key", fact_sales.index + 1)

print(f"fact_sales: {fact_sales.shape}")
fact_sales.head(3)

**Punto clave del role-playing:** el mismo `dim_date` participa **tres veces** en `fact_sales` (`order_date_key`, `required_date_key`, `shipped_date_key`). En pandas no necesitamos resolverlo con tres merges porque la smart key es derivable — pero conceptualmente sigue siendo el patrón Kimball de *role-playing dimension* que viste en el Tema 02.

**`shipped_date_key` con `<NA>`:** la columna usa el tipo `Int64` (nullable), el equivalente pandas del `INT NULL` de SQL. Cuando `shipped_date` es `NaT` (pedido no enviado), `shipped_date_key` queda como `<NA>` — que es exactamente lo que el DDL espera (`shipped_date_key INT REFERENCES dim_date(date_key)`, sin `NOT NULL`).

## Validación de integridad referencial

Antes de cargar al DWH, validamos integridad **en memoria** — es muchísimo más barato detectar un problema aquí que después de un `to_sql` que mueve 2 155 filas a Aurora y rompe una FK a mitad del INSERT.

Las cinco verificaciones clásicas:

1. **Conteo** coincide con el origen (no perdimos ni duplicamos filas).
2. **FKs obligatorias sin nulos** (las que el DDL declara `NOT NULL`).
3. **FKs opcionales con nulos esperados** (las que el DDL permite `NULL`).
4. **Surrogate keys únicas** en cada dimensión.
5. **Spot check de medidas** — el total en pandas debe ser muy cercano al cálculo directo en OLTP.

In [ ]:
# 1. Conteo coincide con el origen
assert len(fact_sales) == len(df_order_details), (
    f"len(fact_sales)={len(fact_sales)} vs len(order_details)={len(df_order_details)}"
)
print(f"✓ fact_sales tiene {len(fact_sales):,} filas (igual que order_details)")

# 2. FKs obligatorias no NULL
fks_obligatorias = ["customer_key", "product_key", "employee_key",
                    "order_date_key", "required_date_key"]
for col in fks_obligatorias:
    nulos = fact_sales[col].isna().sum()
    assert nulos == 0, f"{col} tiene {nulos} NULLs (debería ser 0)"
print(f"✓ FKs obligatorias sin NULLs: {fks_obligatorias}")

# 3. FKs opcionales — reportar (esperado en pedidos no enviados)
for col in ["shipper_key", "shipped_date_key"]:
    nulos = fact_sales[col].isna().sum()
    print(f"  {col}: {nulos} NULLs (esperado en pedidos no enviados)")

# 4. Surrogate keys únicas en cada dimensión
for nombre, dim, key in [
    ("dim_customer", dim_customer, "customer_key"),
    ("dim_product",  dim_product,  "product_key"),
    ("dim_employee", dim_employee, "employee_key"),
    ("dim_shipper",  dim_shipper,  "shipper_key"),
    ("dim_date",     dim_date,     "date_key"),
]:
    assert dim[key].is_unique, f"{nombre}.{key} tiene duplicados"
print(f"✓ Surrogate keys únicas en las 5 dimensiones")

# 5. Spot check de medidas — SUM(line_total) pandas vs OLTP
total_pandas = fact_sales["line_total"].sum()
total_oltp_raw = (
    df_order_details["quantity"]
    * df_order_details["unit_price"]
    * (1 - df_order_details["discount"])
).sum()
print(f"\n  SUM(line_total) pandas: {total_pandas:,.2f}")
print(f"  SUM(line_total) OLTP:   {total_oltp_raw:,.2f}")
print(f"  Diferencia:             {abs(total_pandas - total_oltp_raw):,.4f}")
print("  (la diferencia de centavos es ESPERADA por el redondeo a NUMERIC)")

**`assert` en notebooks de ETL — no es una prueba unitaria, es una invariante de integridad.** Si una falla, no quieres seguir cargando datos rotos al DWH. En un script productivo, estos asserts subirían como excepciones que abortan el ETL y obligan a investigar la causa antes de continuar.

El patrón es deliberadamente **fail-fast**: detectar el problema lo más cerca posible de su origen, donde el contexto para diagnosticarlo todavía está vivo en memoria.

## Cierre

Tienes seis DataFrames en memoria, todos validados y listos para cargar al DWH:

| DataFrame | Filas esperadas | Notas |
|---|---|---|
| `dim_customer` | 91 | Mapeo directo desde `customers` |
| `dim_product` | 77 | Aplanada con `categories` + `suppliers` |
| `dim_employee` | 9 | Self-merge para `reports_to_name`; CEO preservado |
| `dim_shipper` | 6 | Mapeo directo desde `shippers` |
| `dim_date` | 1 096 | Generada sintéticamente, 1996-01-01 a 1998-12-31 |
| `fact_sales` | 2 155 | Tres FKs role-playing a `dim_date`; `order_id` como degenerate dimension |

El siguiente notebook (**04 — Carga y orquestación**) toma estos DataFrames, los carga al schema `northwind_dwh` en Aurora con `to_sql`, y empaqueta todo el pipeline en un script productivo que puede ejecutarse de punta a punta sin intervención manual.

---

<p align="center">
<a href="02_limpieza_y_perfilado.ipynb">← Anterior: Notebook 02</a> | <a href="Readme.md">Volver al índice</a> | <a href="04_carga_y_orquestacion.ipynb">Siguiente: Notebook 04 — Carga y orquestación →</a>
</p>